# Export and classify all NVARC synthetic puzzles on Kaggle

This tutorial exports all 103,223 published NVARC base puzzles. It preserves `nvarc_training` and `nvarc_full`, classifies both parent task IDs against pinned ARC-AGI-1 and ARC-AGI-2 key sets, and writes one portable archive plus task views, key inventories, a provenance index, and a manifest.

**Audience:** Diffusion-ARC data preparation in the Kaggle Notebook web interface.

**Prerequisites:**

- Create a Kaggle Notebook with a CPU session.
- Add [`sorokin/nvarc-synthetic-puzzles`](https://www.kaggle.com/datasets/sorokin/nvarc-synthetic-puzzles), version 1, as a notebook input.
- Enable Internet access for two read-only requests to pinned ARC-AGI-1 and ARC-AGI-2 Git trees. No ARC grids are downloaded from GitHub.
- Leave enough working storage for the compressed export.

**Learning goals:** export every task once, recover parent provenance, materialize the `nvarc_full`, `nvarc_train`, and overlapping `nvarc_train_1` views, and produce hash-verified artifacts for RunPod or AWS.

The 47.5 GB `nvarc-artifacts-puzzles` dataset is not used. The approximately 5.79 GB synthetic-puzzle input remains mounted read-only under `/kaggle/input`.


## Outline

1. Declare and verify pinned sources.
2. Load ARC-AGI-1 and ARC-AGI-2 training/evaluation keys.
3. Classify every NVARC task and preview all views.
4. Validate and archive all task JSONs.
5. Re-read the archive and verify every task hash.
6. Inspect the generated views and transfer the export.


## 1. Pinned sources and source preflight

The expected counts and byte total identify Kaggle dataset version 1. Both current and legacy Kaggle input mount layouts are detected automatically. Existing named outputs are rejected instead of overwritten.


In [ ]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import re
import tarfile
import tempfile
import urllib.error
import urllib.request

NVARC_DATASET_REF = 'sorokin/nvarc-synthetic-puzzles'
NVARC_DATASET_VERSION = 1
NVARC_SUBSETS = ('nvarc_training', 'nvarc_full')
NVARC_ROOT_CANDIDATES = (
    Path('/kaggle/input/datasets/sorokin/nvarc-synthetic-puzzles'),
    Path(
        '/kaggle/input/datasets/sorokin/'
        'nvarc-synthetic-puzzles/versions/1'
    ),
    Path('/kaggle/input/nvarc-synthetic-puzzles'),
)
NVARC_ROOT = next(
    (
        root for root in NVARC_ROOT_CANDIDATES
        if all((root / subset).is_dir() for subset in NVARC_SUBSETS)
    ),
    None,
)
EXPECTED_SOURCE_COUNTS = {'nvarc_training': 47_337, 'nvarc_full': 55_886}
EXPECTED_SOURCE_BYTES = 5_792_311_411
ARC_SOURCES = {
    'arc_agi_1': {
        'repository': 'https://github.com/fchollet/ARC-AGI',
        'commit': '399030444e0ab0cc8b4e199870fb20b863846f34',
        'tree_url': (
            'https://api.github.com/repos/fchollet/ARC-AGI/git/trees/'
            '399030444e0ab0cc8b4e199870fb20b863846f34?recursive=1'
        ),
        'expected_counts': {'training': 400, 'evaluation': 400},
    },
    'arc_agi_2': {
        'repository': 'https://github.com/arcprize/ARC-AGI-2',
        'commit': 'f3283f727488ad98fe575ea6a5ac981e4a188e49',
        'tree_url': (
            'https://api.github.com/repos/arcprize/ARC-AGI-2/git/trees/'
            'f3283f727488ad98fe575ea6a5ac981e4a188e49?recursive=1'
        ),
        'expected_counts': {'training': 1_000, 'evaluation': 120},
    },
}
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_STEM = 'nvarc-all-classified'
ARCHIVE_PATH = OUTPUT_ROOT / f'{OUTPUT_STEM}.tar.gz'
INDEX_PATH = OUTPUT_ROOT / f'{OUTPUT_STEM}-task-index.jsonl'
MANIFEST_PATH = OUTPUT_ROOT / f'{OUTPUT_STEM}-manifest.json'
KEY_GROUPS = (
    'nvarc_full',
    'nvarc_train',
    'nvarc_train_1',
    'arc_agi_1_training',
    'arc_agi_1_evaluation',
    'arc_agi_2_training',
    'arc_agi_2_evaluation',
)
TASK_VIEW_GROUPS = ('nvarc_full', 'nvarc_train', 'nvarc_train_1')
KEY_PATHS = {
    group: OUTPUT_ROOT / f'{group}_keys.json' for group in KEY_GROUPS
}
TASK_VIEW_PATHS = {
    group: OUTPUT_ROOT / f'{group}_tasks.jsonl'
    for group in TASK_VIEW_GROUPS
}
FILENAME_PATTERN = re.compile(
    r'(?P<parent1>[0-9a-f]{8})_(?P<parent2>[0-9a-f]{8})\.json'
)

if NVARC_ROOT is None:
    raise FileNotFoundError(
        f'Add {NVARC_DATASET_REF!r} version {NVARC_DATASET_VERSION} '
        'as a Kaggle Notebook input. Checked: '
        + ', '.join(str(path) for path in NVARC_ROOT_CANDIDATES)
    )
declared_outputs = (
    ARCHIVE_PATH, INDEX_PATH, MANIFEST_PATH,
    *KEY_PATHS.values(), *TASK_VIEW_PATHS.values(),
)
existing_outputs = [path for path in declared_outputs if path.exists()]
if existing_outputs:
    raise FileExistsError(
        'Refusing to overwrite an existing export. Start a fresh session '
        f'or remove only these outputs intentionally: {existing_outputs}'
    )

source_files_by_subset = {}
source_inventory = {}
for subset in NVARC_SUBSETS:
    subset_root = NVARC_ROOT / subset
    if not subset_root.is_dir():
        raise FileNotFoundError(f'Missing NVARC subset: {subset_root}')
    files = tuple(sorted(subset_root.rglob('*.json')))
    if len(files) != EXPECTED_SOURCE_COUNTS[subset]:
        raise RuntimeError(
            f'{subset} count differs: {len(files)} != '
            f'{EXPECTED_SOURCE_COUNTS[subset]}'
        )
    source_files_by_subset[subset] = files
    source_inventory[subset] = {
        'files': len(files),
        'json_bytes': sum(path.stat().st_size for path in files),
    }
actual_source_bytes = sum(item['json_bytes'] for item in source_inventory.values())
if actual_source_bytes != EXPECTED_SOURCE_BYTES:
    raise RuntimeError(
        f'Mounted source bytes differ: {actual_source_bytes} != '
        f'{EXPECTED_SOURCE_BYTES}'
    )
source_inventory


## 2. Load pinned ARC-AGI-1 and ARC-AGI-2 keys

This requests only Git filenames. Membership is recorded independently for both benchmark versions and their training/evaluation splits.


In [ ]:
ARC_PATH_PATTERN = re.compile(
    r'data/(?P<split>training|evaluation)/(?P<task_id>[0-9a-f]{8})\.json'
)


def load_arc_keys(name: str, source: dict[str, object]) -> dict[str, object]:
    request = urllib.request.Request(
        str(source['tree_url']),
        headers={
            'Accept': 'application/vnd.github+json',
            'User-Agent': 'diffusion-arc-nvarc-export/2',
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            tree_payload = json.load(response)
    except urllib.error.URLError as error:
        raise RuntimeError(
            f'Could not read the pinned {name} Git tree. Enable Kaggle Internet.'
        ) from error
    if tree_payload.get('truncated') is not False:
        raise RuntimeError(f'{name} returned a truncated or malformed tree')

    expected_counts = source['expected_counts']
    ids_by_split = {split: set() for split in expected_counts}
    for item in tree_payload.get('tree', []):
        match = ARC_PATH_PATTERN.fullmatch(item.get('path', ''))
        if item.get('type') == 'blob' and match is not None:
            ids_by_split[match.group('split')].add(match.group('task_id'))
    actual_counts = {split: len(ids) for split, ids in ids_by_split.items()}
    if actual_counts != expected_counts:
        raise RuntimeError(
            f'{name} counts differ: {actual_counts} != {expected_counts}'
        )
    if ids_by_split['training'] & ids_by_split['evaluation']:
        raise RuntimeError(f'{name} training and evaluation splits overlap')
    split_by_id = {
        task_id: split
        for split, task_ids in ids_by_split.items()
        for task_id in task_ids
    }
    key_lines = [
        f'{task_id}\t{split_by_id[task_id]}' for task_id in sorted(split_by_id)
    ]
    return {
        'repository': source['repository'],
        'commit': source['commit'],
        'counts': actual_counts,
        'ids_by_split': ids_by_split,
        'split_by_id': split_by_id,
        'key_sha256': hashlib.sha256(
            ('\n'.join(key_lines) + '\n').encode('utf-8')
        ).hexdigest(),
    }


arc_keys = {
    name: load_arc_keys(name, source) for name, source in ARC_SOURCES.items()
}
arc_key_summary = {
    name: {
        'commit': item['commit'],
        'counts': item['counts'],
        'keys': len(item['split_by_id']),
        'key_sha256': item['key_sha256'],
    }
    for name, item in arc_keys.items()
}
arc_key_summary


## 3. Classify every task and preview the views

NVARC paths follow `<subset>/<parent1>/<parent1>_<parent2>.json`. No task is filtered out. `nvarc_train_1` is an overlapping view of `nvarc_training` for which both parents occur in public ARC-AGI-1.


In [ ]:
def parse_source_path(path: Path) -> dict[str, str]:
    relative = path.relative_to(NVARC_ROOT)
    if len(relative.parts) != 3:
        raise ValueError(f'Unexpected path depth: {relative}')
    subset, parent_directory, filename = relative.parts
    match = FILENAME_PATTERN.fullmatch(filename)
    if subset not in NVARC_SUBSETS or match is None:
        raise ValueError(f'Unexpected NVARC path: {relative}')
    parent1 = match.group('parent1')
    parent2 = match.group('parent2')
    if parent_directory != parent1:
        raise ValueError(f'Parent directory disagrees: {relative}')
    return {
        'source_subset': subset,
        'source_relative_path': relative.as_posix(),
        'task_key': f'{parent1}_{parent2}',
        'parent1_id': parent1,
        'parent2_id': parent2,
    }


def lineage_class(first: str | None, second: str | None) -> str:
    if first is None and second is None:
        return 'outside'
    if first is None or second is None:
        return 'one_parent_only'
    if first == second == 'training':
        return 'training_training'
    if first == second == 'evaluation':
        return 'evaluation_evaluation'
    return 'training_evaluation_cross'


TASK_VIEW_DEFINITIONS = {
    'nvarc_full': 'All tasks from the published nvarc_full source subset.',
    'nvarc_train': 'All tasks from the published nvarc_training source subset.',
    'nvarc_train_1': (
        'Tasks from nvarc_training for which both parent keys occur in public '
        'ARC-AGI-1 training or evaluation.'
    ),
}
KEY_DEFINITIONS = {
    **TASK_VIEW_DEFINITIONS,
    'arc_agi_1_training': 'Official pinned ARC-AGI-1 training keys.',
    'arc_agi_1_evaluation': 'Official pinned ARC-AGI-1 evaluation keys.',
    'arc_agi_2_training': 'Official pinned ARC-AGI-2 training keys.',
    'arc_agi_2_evaluation': 'Official pinned ARC-AGI-2 evaluation keys.',
}

arc_agi_1_ids = set(arc_keys['arc_agi_1']['split_by_id'])
task_records = []
for subset in NVARC_SUBSETS:
    for path in source_files_by_subset[subset]:
        record = parse_source_path(path)
        parent1 = record['parent1_id']
        parent2 = record['parent2_id']
        arc_1_first = arc_keys['arc_agi_1']['split_by_id'].get(parent1)
        arc_1_second = arc_keys['arc_agi_1']['split_by_id'].get(parent2)
        arc_2_first = arc_keys['arc_agi_2']['split_by_id'].get(parent1)
        arc_2_second = arc_keys['arc_agi_2']['split_by_id'].get(parent2)
        task_sets = [
            'nvarc_train' if subset == 'nvarc_training' else 'nvarc_full'
        ]
        if (
            subset == 'nvarc_training'
            and parent1 in arc_agi_1_ids
            and parent2 in arc_agi_1_ids
        ):
            task_sets.append('nvarc_train_1')
        record.update(
            {
                'archive_path': f"tasks/{record['source_relative_path']}",
                'task_sets': task_sets,
                'parent1_arc_agi_1_split': arc_1_first,
                'parent2_arc_agi_1_split': arc_1_second,
                'parent1_arc_agi_2_split': arc_2_first,
                'parent2_arc_agi_2_split': arc_2_second,
                'arc_agi_1_lineage': lineage_class(arc_1_first, arc_1_second),
                'arc_agi_2_lineage': lineage_class(arc_2_first, arc_2_second),
            }
        )
        task_records.append(record)
task_records.sort(key=lambda item: item['archive_path'])
archive_paths = [item['archive_path'] for item in task_records]
expected_task_count = sum(EXPECTED_SOURCE_COUNTS.values())
if len(task_records) != expected_task_count:
    raise RuntimeError(f'Task count differs: {len(task_records)} != {expected_task_count}')
if len(archive_paths) != len(set(archive_paths)):
    raise RuntimeError('Duplicate archive paths found')

task_views = {
    group: [item for item in task_records if group in item['task_sets']]
    for group in TASK_VIEW_GROUPS
}
key_sets = {
    group: {
        parent_id
        for item in task_views[group]
        for parent_id in (item['parent1_id'], item['parent2_id'])
    }
    for group in TASK_VIEW_GROUPS
}
for arc_name in ('arc_agi_1', 'arc_agi_2'):
    for split in ('training', 'evaluation'):
        key_sets[f'{arc_name}_{split}'] = set(
            arc_keys[arc_name]['ids_by_split'][split]
        )
if set(key_sets) != set(KEY_GROUPS):
    raise RuntimeError('Key group construction differs from the declared outputs')
if not {item['archive_path'] for item in task_views['nvarc_train_1']} <= {
    item['archive_path'] for item in task_views['nvarc_train']
}:
    raise RuntimeError('nvarc_train_1 is not a subset of nvarc_train')

key_payloads = {
    group: {
        'schema_version': 1,
        'group': group,
        'definition': KEY_DEFINITIONS[group],
        'key_count': len(key_sets[group]),
        'keys': sorted(key_sets[group]),
    }
    for group in KEY_GROUPS
}
dry_run_summary = {
    'all_tasks': len(task_records),
    'task_views': {group: len(items) for group, items in task_views.items()},
    'parent_keys': {group: len(keys) for group, keys in key_sets.items()},
    'by_nvarc_subset': dict(sorted(Counter(
        item['source_subset'] for item in task_records
    ).items())),
    'by_arc_agi_1_lineage': dict(sorted(Counter(
        item['arc_agi_1_lineage'] for item in task_records
    ).items())),
    'by_arc_agi_2_lineage': dict(sorted(Counter(
        item['arc_agi_2_lineage'] for item in task_records
    ).items())),
    'uncompressed_json_gb': round(actual_source_bytes / 10**9, 3),
}
dry_run_summary


## 4. Validate and build the export

Review the dry run first. This cell validates every pair-bank and grid contract, then streams raw task bytes directly into a normalized gzip-compressed tar archive. Metadata views do not duplicate task bytes.


In [ ]:
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def validate_grid(grid: object, label: str) -> None:
    if not isinstance(grid, list) or not grid or len(grid) > 30:
        raise TypeError(f'{label} has an invalid height')
    if not all(isinstance(row, list) and row for row in grid):
        raise TypeError(f'{label} rows must be non-empty lists')
    width = len(grid[0])
    if width > 30 or any(len(row) != width for row in grid):
        raise ValueError(f'{label} must be rectangular and at most 30 wide')
    for row in grid:
        for value in row:
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f'{label} contains a non-integer color')
            if value < 0 or value > 9:
                raise ValueError(f'{label} contains a color outside [0, 9]')


def validate_task(payload: object, source: str) -> int:
    if not isinstance(payload, list) or not payload:
        raise TypeError(f'{source} must contain a non-empty pair list')
    for index, pair in enumerate(payload):
        if not isinstance(pair, dict) or set(pair) != {'input', 'output'}:
            raise TypeError(f'{source} pair {index} has unexpected fields')
        validate_grid(pair['input'], f'{source} pair {index} input')
        validate_grid(pair['output'], f'{source} pair {index} output')
    return len(payload)


def add_bytes(archive: tarfile.TarFile, name: str, data: bytes) -> None:
    info = tarfile.TarInfo(name=name)
    info.size = len(data)
    info.mode = 0o644
    info.mtime = info.uid = info.gid = 0
    info.uname = info.gname = ''
    archive.addfile(info, io.BytesIO(data))


def canonical_json_bytes(payload: object) -> bytes:
    return (json.dumps(
        payload, indent=2, sort_keys=True, allow_nan=False
    ) + '\n').encode('utf-8')


def atomic_write(path: Path, data: bytes) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    if temporary.exists():
        raise FileExistsError(f'Stale temporary output: {temporary}')
    temporary.write_bytes(data)
    temporary.replace(path)


def canonical_jsonl_bytes(records: list[dict[str, object]]) -> bytes:
    return ''.join(
        json.dumps(item, sort_keys=True, allow_nan=False) + '\n'
        for item in records
    ).encode('utf-8')


key_bytes_by_group = {
    group: canonical_json_bytes(payload)
    for group, payload in key_payloads.items()
}
key_archive_paths = {
    group: f'metadata/keys/{group}_keys.json' for group in KEY_GROUPS
}
task_view_archive_paths = {
    group: f'metadata/views/{group}_tasks.jsonl'
    for group in TASK_VIEW_GROUPS
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(
    prefix=f'.{OUTPUT_STEM}-', suffix='.tar.gz.tmp',
    dir=OUTPUT_ROOT, delete=False,
) as handle:
    temporary_archive = Path(handle.name)

export_records = []
try:
    with temporary_archive.open('wb') as raw_output:
        with gzip.GzipFile(
            filename='', mode='wb', fileobj=raw_output,
            compresslevel=6, mtime=0,
        ) as compressed_output:
            with tarfile.open(
                fileobj=compressed_output, mode='w|',
                format=tarfile.PAX_FORMAT,
            ) as archive:
                for number, task_record in enumerate(task_records, start=1):
                    relative = task_record['source_relative_path']
                    raw_json = (NVARC_ROOT / relative).read_bytes()
                    try:
                        payload = json.loads(raw_json)
                    except json.JSONDecodeError as error:
                        raise ValueError(f'Invalid JSON: {relative}') from error
                    pair_count = validate_task(payload, relative)
                    add_bytes(archive, task_record['archive_path'], raw_json)
                    exported = dict(task_record)
                    exported.update(
                        pair_count=pair_count, json_bytes=len(raw_json),
                        sha256=sha256_bytes(raw_json),
                    )
                    export_records.append(exported)
                    if number % 1_000 == 0:
                        print(f'Archived {number:,} / {len(task_records):,}')

                index_bytes = canonical_jsonl_bytes(export_records)
                index_sha256 = sha256_bytes(index_bytes)
                export_by_archive_path = {
                    item['archive_path']: item for item in export_records
                }
                task_view_bytes_by_group = {
                    group: canonical_jsonl_bytes([
                        export_by_archive_path[item['archive_path']]
                        for item in task_views[group]
                    ])
                    for group in TASK_VIEW_GROUPS
                }
                arc_source_manifest = {
                    name: {
                        'repository': item['repository'],
                        'commit': item['commit'],
                        'counts': item['counts'],
                        'key_sha256': item['key_sha256'],
                    }
                    for name, item in arc_keys.items()
                }
                archive_manifest = {
                    'schema_version': 1,
                    'source': {
                        'nvarc': {
                            'dataset_ref': NVARC_DATASET_REF,
                            'version': NVARC_DATASET_VERSION,
                            'expected_total_bytes': EXPECTED_SOURCE_BYTES,
                            'inventory': source_inventory,
                        },
                        **arc_source_manifest,
                    },
                    'selection': {
                        'rule': 'export_all_published_nvarc_tasks_once',
                        'included_nvarc_subsets': list(NVARC_SUBSETS),
                        'task_view_definitions': TASK_VIEW_DEFINITIONS,
                    },
                    'tasks': {
                        'count': len(export_records),
                        'json_bytes': sum(item['json_bytes'] for item in export_records),
                        'by_nvarc_subset': dict(sorted(Counter(
                            item['source_subset'] for item in export_records
                        ).items())),
                        'task_views': {
                            group: len(items) for group, items in task_views.items()
                        },
                        'by_arc_agi_1_lineage': dict(sorted(Counter(
                            item['arc_agi_1_lineage'] for item in export_records
                        ).items())),
                        'by_arc_agi_2_lineage': dict(sorted(Counter(
                            item['arc_agi_2_lineage'] for item in export_records
                        ).items())),
                    },
                    'keys': {
                        group: {
                            'definition': KEY_DEFINITIONS[group],
                            'count': len(key_sets[group]),
                        }
                        for group in KEY_GROUPS
                    },
                    'embedded_files': {
                        'task_index': {
                            'path': 'metadata/task_index.jsonl',
                            'sha256': index_sha256,
                        },
                        'keys': {
                            group: {
                                'path': key_archive_paths[group],
                                'sha256': sha256_bytes(key_bytes_by_group[group]),
                            }
                            for group in KEY_GROUPS
                        },
                        'task_views': {
                            group: {
                                'path': task_view_archive_paths[group],
                                'sha256': sha256_bytes(
                                    task_view_bytes_by_group[group]
                                ),
                            }
                            for group in TASK_VIEW_GROUPS
                        },
                        'archive_manifest': {
                            'path': 'metadata/archive_manifest.json',
                        },
                    },
                    'warnings': [
                        'nvarc_full uses ARC-AGI-2 evaluation-derived descriptions.',
                        'nvarc_train_1 includes both ARC-AGI-1 training and evaluation parents.',
                        'Benchmark evaluation-derived parents are not leakage-safe training sources.',
                        'The published Kaggle dataset reports its license as Unknown.',
                        'Task JSONs are labelled pair banks, not canonical train/test splits.',
                    ],
                }
                archive_manifest_bytes = canonical_json_bytes(archive_manifest)
                add_bytes(archive, 'metadata/task_index.jsonl', index_bytes)
                for group in KEY_GROUPS:
                    add_bytes(
                        archive, key_archive_paths[group], key_bytes_by_group[group]
                    )
                for group in TASK_VIEW_GROUPS:
                    add_bytes(
                        archive, task_view_archive_paths[group],
                        task_view_bytes_by_group[group],
                    )
                add_bytes(archive, 'metadata/archive_manifest.json', archive_manifest_bytes)
    temporary_archive.replace(ARCHIVE_PATH)
finally:
    if temporary_archive.exists():
        temporary_archive.unlink()

archive_sha256 = sha256_file(ARCHIVE_PATH)
external_outputs = {
    'archive': {
        'filename': ARCHIVE_PATH.name,
        'bytes': ARCHIVE_PATH.stat().st_size,
        'sha256': archive_sha256,
    },
    'task_index': {
        'filename': INDEX_PATH.name,
        'bytes': len(index_bytes),
        'sha256': index_sha256,
    },
    'keys': {
        group: {
            'filename': KEY_PATHS[group].name,
            'bytes': len(key_bytes_by_group[group]),
            'sha256': sha256_bytes(key_bytes_by_group[group]),
        }
        for group in KEY_GROUPS
    },
    'task_views': {
        group: {
            'filename': TASK_VIEW_PATHS[group].name,
            'bytes': len(task_view_bytes_by_group[group]),
            'sha256': sha256_bytes(task_view_bytes_by_group[group]),
        }
        for group in TASK_VIEW_GROUPS
    },
}
external_manifest = {
    **archive_manifest,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'outputs': external_outputs,
}
atomic_write(INDEX_PATH, index_bytes)
for group in KEY_GROUPS:
    atomic_write(KEY_PATHS[group], key_bytes_by_group[group])
for group in TASK_VIEW_GROUPS:
    atomic_write(TASK_VIEW_PATHS[group], task_view_bytes_by_group[group])
atomic_write(MANIFEST_PATH, canonical_json_bytes(external_manifest))
if hasattr(os, 'sync'):
    os.sync()
{
    'tasks': len(export_records),
    'archive_gb': round(ARCHIVE_PATH.stat().st_size / 10**9, 3),
    'archive_sha256': archive_sha256,
    'archive': str(ARCHIVE_PATH),
    'task_index': str(INDEX_PATH),
    'key_files': {group: str(path) for group, path in KEY_PATHS.items()},
    'task_view_files': {
        group: str(path) for group, path in TASK_VIEW_PATHS.items()
    },
    'manifest': str(MANIFEST_PATH),
}


## 5. Re-read and verify every archived task

This independent sequential pass verifies the archive member set, every task hash, and all embedded metadata. Expect a second read of the compressed archive.


In [ ]:
verified_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
if verified_manifest != external_manifest:
    raise RuntimeError('External manifest differs from the in-memory manifest')
if sha256_file(ARCHIVE_PATH) != verified_manifest['outputs']['archive']['sha256']:
    raise RuntimeError('Archive hash differs from the manifest')

verified_index_bytes = INDEX_PATH.read_bytes()
if sha256_bytes(verified_index_bytes) != (
    verified_manifest['outputs']['task_index']['sha256']
):
    raise RuntimeError('Task index hash differs from the manifest')
verified_records = [
    json.loads(line) for line in verified_index_bytes.decode('utf-8').splitlines()
    if line
]
if len(verified_records) != expected_task_count:
    raise RuntimeError('Task index does not contain every published task')
if dict(Counter(item['source_subset'] for item in verified_records)) != (
    EXPECTED_SOURCE_COUNTS
):
    raise RuntimeError('Task index source counts differ')

verified_key_bytes = {}
for group in KEY_GROUPS:
    data = KEY_PATHS[group].read_bytes()
    descriptor = verified_manifest['outputs']['keys'][group]
    if sha256_bytes(data) != descriptor['sha256']:
        raise RuntimeError(f'External key hash differs: {group}')
    payload = json.loads(data)
    if (
        data != key_bytes_by_group[group]
        or payload['group'] != group
        or payload['key_count'] != len(payload['keys'])
        or payload['keys'] != sorted(set(payload['keys']))
        or set(payload['keys']) != key_sets[group]
    ):
        raise RuntimeError(f'External key inventory differs: {group}')
    verified_key_bytes[group] = data

verified_task_view_bytes = {}
verified_task_views = {}
for group in TASK_VIEW_GROUPS:
    data = TASK_VIEW_PATHS[group].read_bytes()
    descriptor = verified_manifest['outputs']['task_views'][group]
    if sha256_bytes(data) != descriptor['sha256']:
        raise RuntimeError(f'External task-view hash differs: {group}')
    records = [
        json.loads(line) for line in data.decode('utf-8').splitlines() if line
    ]
    if data != task_view_bytes_by_group[group] or any(
        group not in item['task_sets'] for item in records
    ):
        raise RuntimeError(f'External task view differs: {group}')
    verified_task_view_bytes[group] = data
    verified_task_views[group] = records
rederived_task_view_paths = {
    'nvarc_full': {
        item['archive_path'] for item in verified_records
        if item['source_subset'] == 'nvarc_full'
    },
    'nvarc_train': {
        item['archive_path'] for item in verified_records
        if item['source_subset'] == 'nvarc_training'
    },
    'nvarc_train_1': {
        item['archive_path'] for item in verified_records
        if item['source_subset'] == 'nvarc_training'
        and item['parent1_arc_agi_1_split'] is not None
        and item['parent2_arc_agi_1_split'] is not None
    },
}
for group, records in verified_task_views.items():
    actual_paths = [item['archive_path'] for item in records]
    if (
        len(actual_paths) != len(set(actual_paths))
        or set(actual_paths) != rederived_task_view_paths[group]
    ):
        raise RuntimeError(f'Task view violates its definition: {group}')

expected_task_hashes = {
    item['archive_path']: item['sha256'] for item in verified_records
}
if len(expected_task_hashes) != expected_task_count:
    raise RuntimeError('Task index contains duplicate archive paths')
expected_metadata = {
    'metadata/task_index.jsonl': verified_index_bytes,
    'metadata/archive_manifest.json': archive_manifest_bytes,
    **{
        key_archive_paths[group]: verified_key_bytes[group]
        for group in KEY_GROUPS
    },
    **{
        task_view_archive_paths[group]: verified_task_view_bytes[group]
        for group in TASK_VIEW_GROUPS
    },
}
expected_names = set(expected_task_hashes) | set(expected_metadata)
seen_names = set()
verified_tasks = 0
with tarfile.open(ARCHIVE_PATH, mode='r|gz') as archive:
    for member in archive:
        if not member.isfile() or member.name in seen_names:
            raise RuntimeError(f'Invalid archive member: {member.name}')
        seen_names.add(member.name)
        source = archive.extractfile(member)
        if source is None:
            raise RuntimeError(f'Cannot read: {member.name}')
        data = source.read()
        if member.name in expected_task_hashes:
            if sha256_bytes(data) != expected_task_hashes[member.name]:
                raise RuntimeError(f'Task hash differs: {member.name}')
            verified_tasks += 1
            if verified_tasks % 1_000 == 0:
                print(f'Verified {verified_tasks:,} / {len(verified_records):,}')
        elif member.name in expected_metadata:
            if data != expected_metadata[member.name]:
                raise RuntimeError(f'Metadata differs: {member.name}')
        else:
            raise RuntimeError(f'Unexpected member: {member.name}')
if seen_names != expected_names or verified_tasks != expected_task_count:
    raise RuntimeError('Archive inventory differs from the index')
{
    'verified_tasks': verified_tasks,
    'verified_members': len(seen_names),
    'task_views': {
        group: len(records) for group, records in verified_task_views.items()
    },
    'key_groups': {
        group: len(keys) for group, keys in key_sets.items()
    },
    'status': 'complete',
}


## Exercise: inspect the three requested NVARC views

Confirm that `nvarc_train_1` is contained in `nvarc_train`, while `nvarc_full` is a separate source subset. The answer uses only the verified metadata; task bytes remain stored once.


In [ ]:
view_archive_paths = {
    group: {item['archive_path'] for item in records}
    for group, records in verified_task_views.items()
}
if not view_archive_paths['nvarc_train_1'] <= view_archive_paths['nvarc_train']:
    raise RuntimeError('nvarc_train_1 is not contained in nvarc_train')
if view_archive_paths['nvarc_full'] & view_archive_paths['nvarc_train']:
    raise RuntimeError('Source-subset archive paths unexpectedly overlap')
{
    'task_counts': {
        group: len(paths) for group, paths in view_archive_paths.items()
    },
    'parent_key_counts': {
        group: len(key_sets[group]) for group in KEY_GROUPS
    },
    'nvarc_train_1_is_subset': True,
}


## Download and transfer to RunPod or AWS

After verification succeeds, use **Save Version** in Kaggle so the `/kaggle/working` files become notebook outputs. Download the archive and manifest; the external task index, three task views, and seven key files are convenient for inspection without extracting the archive. On a VM, compare the archive SHA-256 with `outputs.archive.sha256` in the manifest, then extract it with:

```bash
tar -xzf nvarc-all-classified.tar.gz
```

The archive stores all task bytes exactly once under `tasks/nvarc_training/` and `tasks/nvarc_full/`. Its embedded `metadata/task_index.jsonl`, `metadata/views/`, and `metadata/keys/` files are identical to the separately downloadable metadata outputs.

## Common pitfalls and optional extension

- Attach version 1 of `nvarc-synthetic-puzzles`, not the 47.5 GB artifacts dataset.
- Enable Kaggle Internet for the two pinned Git tree requests.
- `nvarc_full` contains ARC-AGI-2 evaluation-derived descriptions. Keep its provenance and exclude it whenever it contaminates evaluation.
- `nvarc_train_1` is an overlapping view, not a third published NVARC source subset. It includes ARC-AGI-1 training and evaluation parent IDs.
- ARC-AGI-1 or ARC-AGI-2 evaluation-derived parents are not leakage-safe training sources.
- Each NVARC JSON is a labelled pair bank, not a canonical `train`/`test` task.
- The published Kaggle dataset reports its license as `Unknown`; public availability does not imply unrestricted redistribution rights.

**Optional extension:** create a separate RunPod/AWS importer that downloads the saved Kaggle output, verifies the manifest, and atomically installs it under provider-specific durable storage.
